# Memory Spill & OOM: Window Functions vs Broadcast Joins

Two of the most common Spark performance killers in production:

1. **Unbounded Window functions** without `partitionBy` — Spark sorts the entire dataset
   in a single partition, causing memory spill or OOM.
2. **SortMergeJoin on tiny dimension tables** — when `autoBroadcastJoinThreshold` is
   disabled or too low, Spark shuffles both sides of a join that should be a cheap
   BroadcastHashJoin.

This notebook shows both anti-patterns, reads their explain plans, and demonstrates the
correct fixes.

In [ ]:
!pip install pyspark --quiet

## Generate Synthetic Data

- **Fact table**: 500K transaction rows (user_id, event_ts, event_type, amount, category_id).
- **Dimension table**: 1 000 categories (category_id, category_name, region).

We deliberately set `spark.driver.memory` to 1g and disable `autoBroadcastJoinThreshold`
to make the problems visible at small scale.

In [ ]:
import time
import random
from datetime import datetime, timedelta
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("WindowBroadcastAntiPatterns")
    .config("spark.driver.memory", "1g")
    .config("spark.sql.autoBroadcastJoinThreshold", "-1")  # disable auto-broadcast
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

random.seed(42)

# --- Fact table: 500K rows ---
N_FACTS = 500_000
base_ts = datetime(2024, 1, 1)

fact_data = [
    (
        random.randint(1, 10_000),                              # user_id
        (base_ts + timedelta(seconds=random.randint(0, 31_536_000))).isoformat(),  # event_ts
        random.choice(["click", "purchase", "view", "cart"]),    # event_type
        round(random.uniform(1, 500), 2),                       # amount
        random.randint(1, 1_000),                               # category_id
    )
    for _ in range(N_FACTS)
]

fact_df = spark.createDataFrame(fact_data, ["user_id", "event_ts", "event_type", "amount", "category_id"])
fact_df = fact_df.withColumn("event_ts", F.to_timestamp("event_ts"))
fact_df.cache()
fact_df.count()  # materialize the cache

# --- Dimension table: 1 000 rows ---
dim_data = [
    (i, f"Category_{i}", random.choice(["US", "EU", "APAC", "LATAM"]))
    for i in range(1, 1_001)
]
dim_df = spark.createDataFrame(dim_data, ["category_id", "category_name", "region"])

print(f"Fact table:      {fact_df.count():>10} rows")
print(f"Dimension table: {dim_df.count():>10} rows")
fact_df.show(5, truncate=False)

## Anti-Pattern 1: Unbounded Window Without `partitionBy`

A common mistake: using `Window.orderBy("event_ts")` **without** `partitionBy`.
This forces Spark to collect the entire dataset into a **single partition** for sorting.
With a large dataset, this causes memory spill to disk (slow) or outright OOM.

In [ ]:
# BAD: no partitionBy — entire dataset in one partition
bad_window = Window.orderBy("event_ts")

print("=== Physical Plan (Anti-Pattern: no partitionBy) ===")
bad_result = fact_df.withColumn("row_num", F.row_number().over(bad_window))
bad_result.explain(True)

print("\nLook for 'SinglePartition' in the plan above.")
print("All 500K rows are sorted in a SINGLE partition — OOM risk on real data.")

In [ ]:
# Time it (safe at 500K but imagine 500M)
start = time.time()
bad_count = bad_result.count()
bad_time = time.time() - start
print(f"Anti-pattern 1 execution: {bad_time:.2f}s for {bad_count} rows")

## Anti-Pattern 2: SortMergeJoin on a Tiny Dimension Table

With `autoBroadcastJoinThreshold` disabled (or set too low, which happens in
hardened production configs), Spark defaults to **SortMergeJoin** even when one
side is tiny. This shuffles both sides across the network — completely unnecessary
for a 1 000-row lookup table.

In [ ]:
# BAD: SortMergeJoin (because autoBroadcast is disabled)
bad_join = fact_df.join(dim_df, on="category_id")

print("=== Physical Plan (Anti-Pattern: SortMergeJoin) ===")
bad_join.explain(True)

print("\nLook for 'SortMergeJoin' and 'Exchange hashpartitioning' in the plan.")
print("Both sides are shuffled — the 1K-row dimension table is shuffled for nothing.")

In [ ]:
start = time.time()
smj_count = bad_join.count()
smj_time = time.time() - start
print(f"SortMergeJoin execution: {smj_time:.2f}s for {smj_count} rows")

## Fix 1: Partition Your Windows

Adding `partitionBy("user_id")` distributes the work across partitions. Each partition
contains only that user's rows, keeping memory usage bounded and enabling parallelism.

In [ ]:
# GOOD: partition by user_id
good_window = Window.partitionBy("user_id").orderBy("event_ts")

good_result = fact_df.withColumn("row_num", F.row_number().over(good_window))

print("=== Physical Plan (Fix: with partitionBy) ===")
good_result.explain(True)

print("\nNow look for 'hashpartitioning(user_id)' — work is distributed across partitions.")

start = time.time()
good_count = good_result.count()
good_time = time.time() - start

print(f"\nPartitioned window execution: {good_time:.2f}s for {good_count} rows")
print(f"Speedup vs anti-pattern: {bad_time / good_time:.1f}x")

## Fix 2: Broadcast Small Tables

For small dimension tables (up to ~100MB), use `F.broadcast()` to send the entire
table to every executor. This eliminates the shuffle on the fact side entirely.

In [ ]:
# GOOD: force BroadcastHashJoin
good_join = fact_df.join(F.broadcast(dim_df), on="category_id")

print("=== Physical Plan (Fix: BroadcastHashJoin) ===")
good_join.explain(True)

print("\nLook for 'BroadcastHashJoin' and 'BroadcastExchange' — no shuffle on the fact side.")

start = time.time()
bhj_count = good_join.count()
bhj_time = time.time() - start

print(f"\nBroadcastHashJoin execution: {bhj_time:.2f}s for {bhj_count} rows")
print(f"Speedup vs SortMergeJoin: {smj_time / bhj_time:.1f}x")

## Reading Spark Explain Plans: What to Look For

| Plan Element | What It Means | Good or Bad? |
| :--- | :--- | :--- |
| `SinglePartition` | Entire dataset in one partition | Bad — OOM risk |
| `Exchange hashpartitioning` | Data shuffled across executors | Unavoidable sometimes, but expensive |
| `SortMergeJoin` | Both sides sorted + merged | Bad for small dim tables — use Broadcast instead |
| `BroadcastHashJoin` | Small table sent to all executors | Good for dim tables < 100MB |
| `BroadcastExchange` | The broadcast distribution step | Expected companion of BroadcastHashJoin |

In [ ]:
# Summary comparison
import pandas as pd

summary = pd.DataFrame([
    {"Scenario": "Window: no partitionBy (anti-pattern)", "Time (s)": f"{bad_time:.2f}", "Plan": "SinglePartition sort"},
    {"Scenario": "Window: partitionBy user_id (fix)", "Time (s)": f"{good_time:.2f}", "Plan": "Distributed hash partition"},
    {"Scenario": "Join: SortMergeJoin (anti-pattern)", "Time (s)": f"{smj_time:.2f}", "Plan": "Exchange + Sort both sides"},
    {"Scenario": "Join: BroadcastHashJoin (fix)", "Time (s)": f"{bhj_time:.2f}", "Plan": "Broadcast small table"},
])
display(summary)

spark.stop()

## Takeaways

| Problem | Symptom in Spark UI | Fix | Rule of Thumb |
| :--- | :--- | :--- | :--- |
| Window without `partitionBy` | One task runs for hours, others finish instantly | Add `partitionBy` on a high-cardinality column | Always partition windows unless you intentionally need global ordering |
| SortMergeJoin on small table | Unnecessary shuffle stages on both sides | `F.broadcast(small_df)` | Broadcast any table < 100MB; check `autoBroadcastJoinThreshold` in prod configs |
| Memory spill to disk | Spill metrics > 0 in the Spark UI Stages tab | Increase partition count or add `partitionBy` | If spill > 20% of data size, repartition or rethink the approach |